# Notebook 35 — Durable Agent Workflows, Memory, and Human Approval

    ## Learning objectives

    - Model agents as resumable state machines with idempotent side effects
- Separate working, episodic, semantic, procedural, and user-profile memory
- Design pause/resume, approval, cancellation, retries, and recovery for long-running work

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 35.1 An agent run is a workflow with nondeterministic nodes

A chat loop held in process memory disappears on restart and cannot safely wait hours for approval. Production
work needs explicit state, named transitions, durable checkpoints, and an event history. Model calls are
nondeterministic decision nodes; validation, authorization, tool execution, and state transitions should be
deterministic wherever possible. A graph framework can implement this, but the invariants belong to the system.

Store authoritative facts in typed state rather than reconstructing them from prose. Include run and tenant IDs,
status, step/version, budgets, approved plan version, pending action, completed action receipts, and deadlines.
Optimistic concurrency or serialized ownership prevents two workers from advancing the same run.


In [ ]:
from dataclasses import dataclass, field, asdict
from enum import Enum
class Status(str, Enum): READY="ready"; WAITING="waiting_approval"; RUNNING="running"; DONE="done"; FAILED="failed"; CANCELLED="cancelled"
@dataclass
class RunState:
    run_id: str; tenant_id: str; status: Status = Status.READY; version: int = 0
    remaining_steps: int = 8; pending_action: dict | None = None
    receipts: dict[str, dict] = field(default_factory=dict)
state = RunState("run-42", "tenant-a")
print(asdict(state))


## 35.2 Checkpoints, replay, and side effects

Persist before and after externally visible actions. Retrying a model read is usually wasteful but retrying a
payment, message, or deployment can duplicate harm. Give mutations stable idempotency keys derived from the
approved action—not from a transient attempt—and require downstream tools to return receipts. On resume, inspect
the receipt store before executing.

Event sourcing records immutable proposals, approvals, executions, observations, cancellations, and errors. A
materialized state accelerates reads but can be rebuilt from events. Never replay side effects to rebuild state.
Schema-version checkpoints and migrations; encrypt sensitive fields; enforce tenant access; and set retention.


In [ ]:
import hashlib, json
def action_key(run_id, action):
    canonical = json.dumps(action, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(f"{run_id}:{canonical}".encode()).hexdigest()
action = {"tool":"publish_report", "arguments":{"report_id":"r-7"}}
key = action_key(state.run_id, action)
state.receipts[key] = {"status":"committed", "external_id":"publication-91"}
print("retry decision:", "return receipt" if key in state.receipts else "execute")


## 35.3 Human-in-the-loop is a state transition

Approval should display the exact effect, target, arguments, data leaving the boundary, estimated cost, and
rollback limits. Bind approval to an immutable action hash and expiry. If the model changes an argument, request
new approval. Record approver identity through authenticated application context; text saying “approved” is not
authorization. Support reject, edit, request-more-information, and cancel—not only yes.

An interrupt persists state and yields a user-visible request. Resume supplies a structured decision under
optimistic concurrency. The workflow revalidates current policy because permissions or external state may have
changed while waiting. Timeouts enter an explicit state rather than silently approving or retrying.


In [ ]:
def request_approval(state, action):
    state.pending_action = {**action, "hash": action_key(state.run_id, action), "expires_at":"2026-09-03T00:00:00Z"}
    state.status = Status.WAITING; state.version += 1
    return {"run_id":state.run_id, "version":state.version, "requested":state.pending_action}
print(request_approval(state, {"tool":"send_email", "arguments":{"recipient":"owner@example.invalid"}}))


## 35.4 Memory is several systems

Working memory is the bounded current trajectory. Episodic memory stores past runs and outcomes. Semantic memory
retrieves durable facts. Procedural memory contains instructions and policies. User-profile memory stores approved
preferences. Do not combine them into one vector database. Each has different writers, truth authority, access,
lifetime, deletion, and conflict semantics.

Memory writes are consequential: require provenance, timestamps, tenant/user ownership, confidence, expiry, and
a reason for retention. Prefer application-derived facts over model summaries. Retrieve with authorization filters
applied inside the query, rerank, expose provenance, and let current authoritative records override stale memory.
Poisoned tool output or prompt injection must not become permanent policy.


In [ ]:
memories = [
    {"kind":"profile", "fact":"prefers concise reports", "source":"user_setting", "expires":None},
    {"kind":"episodic", "fact":"deployment failed health check", "source":"run-39", "expires":"2026-10-01"},
    {"kind":"semantic", "fact":"service owner is team-x", "source":"catalog@17", "expires":"2026-09-03"},
]
writable_by_model = {"working"}
for memory in memories: print(memory["kind"], "model may directly write:", memory["kind"] in writable_by_model)


## 35.5 Context engineering and compaction

Build context from typed state, current task, relevant observations, policies, retrieved memory, and tool schemas
under an explicit token budget. Drop redundant raw traces only after preserving action receipts and unresolved
constraints. Summaries are lossy and potentially incorrect; store links to original events and validate critical
fields separately. Do not expose every tool and memory on every step.

Compaction policies should be deterministic and tested: retain system constraints, open commitments, errors,
approvals, latest results, and citations; summarize low-priority dialogue; evict obsolete observations. Measure
success, prompt tokens, forgotten-constraint rate, and injection persistence before adopting a memory strategy.


In [ ]:
events = [{"kind":"constraint","text":"never publish without approval","priority":100},
          {"kind":"observation","text":"draft has 1200 words","priority":40},
          {"kind":"chat","text":"thanks","priority":1}]
budget_items = 2
selected = sorted(events, key=lambda event:event["priority"], reverse=True)[:budget_items]
print("context:", selected)


## 35.6 Cancellation, faults, and framework mapping

Cancellation must propagate to queued model calls and tools, while acknowledging that an external side effect may
already have committed. Distinguish transient transport errors, invalid model actions, policy denials, tool-domain
errors, and permanent failures. Retry only classified transient operations with bounded attempts and jitter.
Compensating actions are domain workflows, not automatic rollback.

LangGraph-style checkpointers and interrupts are one implementation of durable pause/resume; workflow engines and
databases can provide the same invariants. Test crash points before/after every state write and side effect. Replay
deterministic nodes from captured events, inject worker loss, expire approvals, change permissions during waits,
and verify exactly-once effect semantics or explicitly documented at-least-once behavior.


## Exercises

    1. Implement a crash-safe two-phase tool action with an idempotency receipt.
2. Design memory schemas and deletion policies for all five memory categories.
3. Test an approval that becomes stale after its proposed arguments change.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
